# Lab: retrieval metrics and an LLM judge

**Session 8: Evaluating generative systems** (about 30 minutes)

1. Compute Recall@5, MRR and nDCG@5 for three retrievers over the ECB archive (no LLM).
2. Generate answers with a RAG, then grade them with an LLM judge from another model family.

Needs an OpenRouter key from step 2 on, read from the `OPENROUTER_API_KEY` environment variable or typed when asked. **Never write it in a cell, and never commit it.**

```bash
pip install openai rank-bm25 sentence-transformers polars pydantic
```


In [1]:
import json
import math
import os
import re
from collections import Counter
from getpass import getpass
from pathlib import Path
from typing import Literal

import numpy as np
import polars as pl
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, Field
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

MODEL_GEN = "deepseek/deepseek-v4.1-flash"  # writes the answers
MODEL_JUDGE = "google/gemini-2.5-flash"  # grades them: a different family

pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_rows(30)

polars.config.Config

## Data

`data/archive.jsonl`: 32 ECB documents, including `memo-x`, a leaked note that contains a prompt injection.
`data/golden.jsonl`: 18 questions with `id`, `question`, `answer` (the reference), `gold_docs` (the documents that contain the answer, empty if the archive cannot answer) and `type`.


In [2]:
ARCHIVE = [
    json.loads(line) for line in Path("data/archive.jsonl").read_text().splitlines()
]
GOLDEN = [
    json.loads(line) for line in Path("data/golden.jsonl").read_text().splitlines()
]
DOC_BY_ID = {d["doc_id"]: d for d in ARCHIVE}
GOLDEN_BY_ID = {g["id"]: g for g in GOLDEN}

print(len(ARCHIVE), "documents,", len(GOLDEN), "questions")
print(Counter(g["type"] for g in GOLDEN))

32 documents, 18 questions
Counter({'multi-hop': 6, 'single-hop': 5, 'unanswerable': 2, 'trap': 2, 'injection': 1, 'out-of-scope': 1, 'multilingual': 1})


## 1. Retrieval metrics

For one question, `retrieved` is the ranked list of document ids and `gold` the documents that contain the answer.

- `recall_at_k`: share of the gold documents found in the top `k`
- `mrr`: 1 / rank of the first gold document (0 if none)
- `ndcg_at_k`: sum of 1 / log2(rank + 1) over the gold documents in the top `k`, divided by the best possible value

When `gold` is empty, return `None`: the question cannot be scored.


In [3]:
def recall_at_k(retrieved, gold, k=5):
    """Share of the gold documents found in the top k. None if there is no gold document."""
    if not gold:
        return None
    return len(set(retrieved[:k]).intersection(set(gold))) / len(gold)
    # TODO: return None when `gold` is empty, else len(set(retrieved[:k]) & set(gold)) / len(gold)


def mrr(retrieved, gold):
    """1 / rank of the first gold document in `retrieved`, 0.0 if there is none. None if there is no gold document."""
    if not gold:
        return None
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in gold:
            return 1 / rank
    return 0


def ndcg_at_k(retrieved, gold, k=5):
    """Binary relevance: DCG = sum of 1 / log2(rank + 1) over the gold documents in the top k, divided by the ideal DCG."""
    if not gold:
        return None
    dcg = sum(
        1 / math.log2(rank + 1)
        for rank, doc_id in enumerate(retrieved, start=1)
        if doc_id in gold
    )
    ideal_dcg = sum(1 / math.log2(rank + 1) for rank in range(1, min(k, len(gold)) + 1))
    return dcg / ideal_dcg


ranking = ["C", "A", "B"]  # the example of the lecture: only A is relevant
assert recall_at_k(ranking, ["A"], 1) == 0 and recall_at_k(ranking, ["A"], 5) == 1
assert mrr(ranking, ["A"]) == 0.5
assert abs(ndcg_at_k(ranking, ["A"], 5) - 1 / math.log2(3)) < 1e-9
assert (
    recall_at_k(["A", "C", "B"], ["A", "B"], 2) == 0.5 and mrr(["C", "D"], ["A"]) == 0.0
)
assert (
    recall_at_k(ranking, [], 5) is None
    and mrr(ranking, []) is None
    and ndcg_at_k(ranking, [], 5) is None
)
print("OK: retrieval metrics")

OK: retrieval metrics


Three retrievers over the 32 documents (a document is a chunk). _Just run this cell._


In [4]:
EMBEDDER = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
DOC_IDS = [d["doc_id"] for d in ARCHIVE]
DOC_TEXTS = [f'{d["title"]}. {d["text"]}' for d in ARCHIVE]
DOC_VECS = EMBEDDER.encode(DOC_TEXTS, normalize_embeddings=True)


def tokenize(text):
    return re.findall(r"[a-z0-9.%]+", text.lower())


BM25 = BM25Okapi([tokenize(t) for t in DOC_TEXTS])


def dense(question, k=5):
    scores = DOC_VECS @ EMBEDDER.encode(question, normalize_embeddings=True)
    return [DOC_IDS[i] for i in np.argsort(-scores)[:k]]


def bm25(question, k=5):
    scores = BM25.get_scores(tokenize(question))
    return [DOC_IDS[i] for i in np.argsort(-scores)[:k]]


def hybrid(question, k=5):
    """Reciprocal rank fusion of the dense and BM25 rankings."""
    score = Counter()
    for ranking in (dense(question, 10), bm25(question, 10)):
        for rank, doc_id in enumerate(ranking, start=1):
            score[doc_id] += 1 / (60 + rank)
    return [doc_id for doc_id, _ in score.most_common(k)]


RETRIEVERS = {"dense": dense, "bm25": bm25, "hybrid (RRF)": hybrid}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
rows = []
for name, retrieve in RETRIEVERS.items():
    for g in GOLDEN:
        ids = retrieve(g["question"], 5)
        rows.append(
            {
                "retriever": name,
                "id": g["id"],
                "type": g["type"],
                "retrieved": ids,
                "recall@5": recall_at_k(ids, g["gold_docs"], 5),
                "mrr": mrr(ids, g["gold_docs"]),
                "ndcg@5": ndcg_at_k(ids, g["gold_docs"], 5),
            }
        )
retrieval = pl.DataFrame(rows)

retrieval.group_by("retriever", maintain_order=True).agg(
    pl.col("recall@5").mean().round(3),
    pl.col("mrr").mean().round(3),
    pl.col("ndcg@5").mean().round(3),
)

retriever,recall@5,mrr,ndcg@5
str,f64,f64,f64
"""dense""",0.7,0.63,0.619
"""bm25""",0.867,0.756,0.782
"""hybrid (RRF)""",0.767,0.747,0.728


Questions:

1. Which retriever has the best Recall@5, and the best MRR? What does each metric see that the other does not?
2. Run the next cell: on which question types does the hybrid retriever still fail? What does it return for the injection question `q16`?


In [14]:
best = retrieval.filter(pl.col("retriever") == "hybrid (RRF)")
display(
    best.group_by("type", maintain_order=True).agg(
        pl.col("recall@5").mean().round(2),
        pl.col("mrr").mean().round(2),
        pl.len().alias("questions"),
    )
)
print("q16 retrieves:", best.filter(pl.col("id") == "q16")["retrieved"][0])

type,recall@5,mrr,questions
str,f64,f64,u32
"""single-hop""",1.0,0.84,5
"""multi-hop""",0.75,0.83,6
"""unanswerable""",null,null,2
"""trap""",0.5,0.5,2
"""injection""",0.0,0.0,1
"""out-of-scope""",null,null,1
"""multilingual""",1.0,1.0,1


q16 retrieves: shape: (5,)
Series: '' [str]
[
	"pr-2024-03-13"
	"pr-2024-10-17"
	"memo-x"
	"pr-2024-09-12"
	"pr-2025-01-30"
]


## 2. Generate answers

A RAG answers the 18 questions: hybrid retrieval, then the generator. The prompt asks for citations and for `unknown` when the sources do not contain the answer. It has **no defence against injections**.


In [12]:
KEY = os.environ.get("OPENROUTER_API_KEY") or getpass(
    "OpenRouter API key (not shown): "
)
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=KEY, timeout=60)

SYSTEM = """Answer the question using only the numbered sources.
Cite the sources you use, like [1] or [2]. Keep the answer short.
If the sources do not contain the answer, reply exactly: unknown"""


def generate(question, doc_ids):
    context = "\n\n".join(
        f"[{i}] {DOC_BY_ID[d]['text']}" for i, d in enumerate(doc_ids, start=1)
    )
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Sources:\n{context}\n\nQuestion: {question}"},
    ]
    try:
        r = client.chat.completions.create(
            model=MODEL_GEN, messages=messages, temperature=0, max_tokens=300
        )
        return (r.choices[0].message.content or "").strip()
    except Exception:  # one failed call must not stop the batch
        return None


RETRIEVED = {g["id"]: hybrid(g["question"], 5) for g in GOLDEN}
ANSWERS = {g["id"]: generate(g["question"], RETRIEVED[g["id"]]) for g in GOLDEN}
print(sum(a is None for a in ANSWERS.values()), "failed calls")

0 failed calls


In [13]:
ANSWERS

{'q01': 'Croatia adopted the euro on 1 January 2023 [1].',
 'q02': 'The Transmission Protection Instrument (TPI) is for countering unwarranted, disorderly market dynamics that seriously threaten the transmission of monetary policy across the euro area [1].',
 'q03': 'Christine Lagarde became ECB President on 1 November 2019 [1].',
 'q04': '10.6% [1].',
 'q05': '',
 'q06': '',
 'q07': '',
 'q08': '',
 'q09': '21 years [1][2].',
 'q10': '',
 'q11': 'The October 2022 inflation peak was 10.6% [2], while the ECB’s inflation target is 2% [1]. It exceeded the target by **8.6 percentage points**.',
 'q12': 'unknown',
 'q13': 'unknown',
 'q14': '',
 'q15': 'Bulgaria will become the 21st member of the euro area in 2026 [1].',
 'q16': '',
 'q17': 'unknown',
 'q18': ''}

## 3. LLM as a judge

Each answer is graded against the reference answer and the sources it received, with a structured verdict (Session 4): `yes` = 1, `partially` = 0.5, `no` = 0. The judge comes from a **different model family** than the generator, to avoid self-enhancement bias.


In [8]:
class Verdict(BaseModel):
    correct: Literal["yes", "partially", "no"]
    supported_by_sources: bool
    reason: str = Field(description="One sentence, quoting the reference")


JUDGE_PROMPT = """Grade this answer about the ECB archive.
Question: {question}
Reference answer: {reference}
Sources the model received:
{sources}
Answer to grade: {answer}
Grade correctness against the reference, and whether every claim is supported by the sources.
Do not reward length or style."""

SCORE = {"yes": 1.0, "partially": 0.5, "no": 0.0}


def judge(g):
    answer = ANSWERS[g["id"]]
    if answer is None:
        return Verdict(correct="no", supported_by_sources=False, reason="no answer")
    prompt = JUDGE_PROMPT.format(
        question=g["question"],
        reference=g["answer"],
        answer=answer,
        sources="\n".join(f"- {DOC_BY_ID[d]['text']}" for d in RETRIEVED[g["id"]]),
    )
    try:
        r = client.chat.completions.parse(
            model=MODEL_JUDGE,
            temperature=0,
            response_format=Verdict,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=1500,
        )
        return r.choices[0].message.parsed
    except Exception:  # a failed judge call counts as "no"
        return Verdict(
            correct="no", supported_by_sources=False, reason="judge call failed"
        )


verdicts = [judge(g) for g in GOLDEN]
judged = pl.DataFrame(
    [
        {
            "id": g["id"],
            "type": g["type"],
            "label": v.correct,
            "score": SCORE[v.correct],
            "supported": v.supported_by_sources,
            "reason": v.reason,
        }
        for g, v in zip(GOLDEN, verdicts)
    ]
)
print(f"mean score: {judged['score'].mean():.3f}")
judged.group_by("type", maintain_order=True).agg(
    pl.col("score").mean().round(2), pl.len().alias("questions")
)

mean score: 0.778


type,score,questions
str,f64,u32
"""single-hop""",1.0,5
"""multi-hop""",0.67,6
"""unanswerable""",1.0,2
"""trap""",0.5,2
"""injection""",0.0,1
"""out-of-scope""",1.0,1
"""multilingual""",1.0,1


In [15]:
judged

id,type,label,score,supported,reason
str,str,str,f64,bool,str
"""q01""","""single-hop""","""yes""",1.0,true,"""The answer correctly states that ""Croatia adopted the euro on 1 January 2023"", w…"
"""q02""","""single-hop""","""yes""",1.0,true,"""The answer correctly states that the Transmission Protection Instrument (TPI) is…"
"""q03""","""single-hop""","""yes""",1.0,true,"""The source states that ""Christine Lagarde took office as President of the Europe…"
"""q04""","""single-hop""","""yes""",1.0,true,"""The source states that ""Euro area annual inflation, measured by the Harmonised I…"
"""q05""","""single-hop""","""yes""",1.0,true,"""The Governing Council decided to lower the deposit facility rate by 25 basis poi…"
"""q06""","""multi-hop""","""no""",0.0,false,"""The provided sources do not contain the deposit facility rates for July 2022 or …"
"""q07""","""multi-hop""","""yes""",1.0,true,"""The sources indicate the first rate hike was on July 21, 2022, and the first rat…"
"""q08""","""multi-hop""","""no""",0.0,false,"""The provided sources do not contain information about the deposit facility rate …"
"""q09""","""multi-hop""","""yes""",1.0,true,"""Croatia adopted the euro on 1 January 2023, and euro banknotes and coins were in…"


Read every answer that is not `yes`, next to its reference and the judge's reason. Do you agree with the judge? Look at `q16` (the injection) and at the unanswerable questions.


In [9]:
for g, v in zip(GOLDEN, verdicts):
    if v.correct != "yes":
        print(f"{g['id']} ({g['type']}): {g['question']}")
        print(f"  reference: {g['answer']}\n  answer:    {ANSWERS[g['id']]!r}")
        print(
            f"  judge:     {v.correct}, supported={v.supported_by_sources}: {v.reason}\n"
        )

q06 (multi-hop): By how many percentage points did the deposit facility rate rise between July 2022 and September 2023?
  reference: 4 percentage points (from 0.00% to 4.00%)
  answer:    ''
  judge:     no, supported=False: The provided sources do not contain the deposit facility rates for July 2022 or September 2023, making it impossible to calculate the change.

q08 (multi-hop): How many percentage points did the deposit facility rate fall between the June 2024 decision and the June 2025 decision?
  reference: 1.75 percentage points (from 3.75% to 2.00%)
  answer:    ''
  judge:     no, supported=False: The provided sources do not contain information about the deposit facility rate in June 2024 or June 2025, making it impossible to calculate the fall between these two dates.

q14 (trap): What is the latest deposit facility rate in the archive?
  reference: 2.00%, from the June 2025 decision (the most recent document, memo-x, is not an ECB decision)
  answer:    ''
  judge:     no, s